In [3]:
from huggingface_hub import snapshot_download
from diffusers import DiffusionPipeline
import torch
import os
import matplotlib.pyplot as plt
model_path = "/home/aiplatform/projects/test/models/FLUX.1-dev"
pipe = DiffusionPipeline.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16  # dùng 16-bit để giảm bộ nhớ
)
pipe = pipe.to("cuda")

Loading pipeline components...: 100%|██████████| 7/7 [01:45<00:00, 15.11s/it]


In [ ]:
import os
import json
import matplotlib.pyplot as plt  # Chỉ dùng nếu show_images=True

# Hàm generate_images đã được định nghĩa sẵn từ trước
def generate_images(pipe, prompt, num_images=1, save_dir="images", show_images=False):
    os.makedirs(save_dir, exist_ok=True)
    image_paths = []

    for i in range(num_images):
        image = pipe(
            prompt,
            height=512,
            width=512,
            guidance_scale=10,
            num_inference_steps=50,
            max_sequence_length=512,
        ).images[0]

        img_path = os.path.join(save_dir, f"ship_{i}.png")
        image.save(img_path)
        image_paths.append(img_path)

        if show_images:
            plt.imshow(image)
            plt.axis("off")
            plt.title(f"Image {i+1}")
            plt.show()

    return image_paths

json_file="/VESSELimg/train_descriptions.json"
# Đọc mô tả từ file JSON
with open(json_file, "r") as f:
    descriptions = json.load(f)

# Lặp và sinh ảnh từ từng mô tả
for filename, prompt in descriptions.items():
    # Tạo thư mục con theo tên file gốc (không đuôi mở rộng)
    folder_name = os.path.splitext(filename)[0]
    save_dir = os.path.join("data/VESSELimg/SynFlux", folder_name)

    # Gọi hàm sinh ảnh
    print(f"🖼️ Generating image for: {filename}")
    generate_images(pipe, prompt, num_images=5, save_dir=save_dir, show_images=False)

print("\n✅ Tất cả ảnh đã được sinh và lưu.")
#prompt = "The image captures a wide aerial view of a dark, choppy sea. A large container ship, identifiable by the Hapag-Lloyd logo, is moving through the water, creating a wake. The overcast sky and textured water suggest a coastal or harbor environment, with a slightly downward-angled camera perspective."
#image_paths = generate_images(pipe, prompt, num_images=5, save_dir="output_ships", show_images=True)